In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from tqdm import tqdm

In [2]:
def find_project_root(target_folder="hest1k-dataset"):
    path = os.getcwd()

    while True:
        if target_folder in os.listdir(path):
            return path

        parent = os.path.dirname(path)
        if parent == path:
            raise Exception("Project root not found")

        path = parent


BASE_DIR = find_project_root()
HEST_DATA_DIR = os.path.join(BASE_DIR, "hest1k-dataset", "hest_multicancer")
WSI_DIR = os.path.join(HEST_DATA_DIR, "wsis")
ST_DIR = os.path.join(HEST_DATA_DIR, "st")
META_DIR = os.path.join(HEST_DATA_DIR, "metadata")
PROCESSED_DIR = os.path.join(BASE_DIR, "data300", "processed")

os.makedirs(PROCESSED_DIR, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("DATA_DIR:", HEST_DATA_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("META EXISTS:", os.path.exists(META_DIR))

BASE_DIR: c:\Users\dhanu\Documents\preliminary-gsoc\histMOE
DATA_DIR: c:\Users\dhanu\Documents\preliminary-gsoc\histMOE\hest1k-dataset\hest_multicancer
PROCESSED_DIR: c:\Users\dhanu\Documents\preliminary-gsoc\histMOE\data300\processed
META EXISTS: True


In [3]:
patches = np.load(os.path.join(PROCESSED_DIR, "patches.npy"))
genes   = np.load(os.path.join(PROCESSED_DIR, "genes.npy"))
meta    = np.load(os.path.join(PROCESSED_DIR, "metadata.npy"))
labels  = np.load(os.path.join(PROCESSED_DIR, "labels.npy"))
sample_ids = np.load(os.path.join(PROCESSED_DIR, "sample_ids.npy"))

print(patches.shape, genes.shape)

(19413, 128, 128, 3) (19413, 150)


In [4]:
class HistoDataset(Dataset):
    def __init__(self, patches, genes, meta, labels):
        self.patches = patches
        self.genes = genes
        self.meta = meta
        self.labels = labels

        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, idx):
        image = self.transform(self.patches[idx])

        return (
            image.clone().detach().float(),
            torch.tensor(self.genes[idx], dtype=torch.float32),
            torch.tensor(self.meta[idx], dtype=torch.float32),
            torch.tensor(self.labels[idx], dtype=torch.long),
        )

In [5]:
from sklearn.model_selection import train_test_split

unique_ids = np.unique(sample_ids)

train_ids = ["TENX155", "TENX175", "TENX171", "TENX189"]
val_ids   = ["TENX156","TENX190"]   # COAD
# test_ids  = ["TENX190"]   # LUAD

print("Train:", train_ids)
print("Val:", val_ids)
# print("Test:", test_ids)

train_mask = np.isin(sample_ids, train_ids)
val_mask   = np.isin(sample_ids, val_ids)
# test_mask  = np.isin(sample_ids, test_ids)

train_dataset = HistoDataset(
    patches[train_mask],
    genes[train_mask],
    meta[train_mask],
    labels[train_mask]
)

val_dataset = HistoDataset(
    patches[val_mask],
    genes[val_mask],
    meta[val_mask],
    labels[val_mask]
)

# test_dataset = HistoDataset(
#     patches[test_mask],
#     genes[test_mask],
#     meta[test_mask],
#     labels[test_mask]
# )

Train: ['TENX155', 'TENX175', 'TENX171', 'TENX189']
Val: ['TENX156', 'TENX190']


In [6]:
from torch.utils.data import DataLoader
from torch.utils.data import WeightedRandomSampler

class_counts = np.bincount(labels[train_mask])
weights = 1. / class_counts

sample_weights = weights[labels[train_mask]]

sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    sampler=sampler,
    num_workers=0
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    pin_memory=True
)

# test_loader = DataLoader(
#     test_dataset,
#     batch_size=32,
#     shuffle=False)

In [7]:
class MetadataEncoder(nn.Module):
    def __init__(self, input_dim=7, embed_dim=64):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, embed_dim)
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
from torchvision.models import resnet18

class MoE(nn.Module):
    def __init__(self, num_experts=2, gene_dim=300):
        super().__init__()

        # ---- Image encoder ----
        self.encoder = resnet18(pretrained=True)
        self.encoder.fc = nn.Identity()   # 512-d output
        for name, param in self.encoder.named_parameters():
            if "layer4" in name:
                param.requires_grad = True
            else:
                param.requires_grad = False

        # ---- Metadata encoder ----
        self.meta_encoder = MetadataEncoder(input_dim=7)

        # ---- Gating ----
        self.gate = nn.Sequential(
            nn.Linear(512 + 64, 128),
            nn.ReLU(),
            nn.Linear(128, num_experts)
        )

        # ---- Experts ----
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(512+64, 512),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(512, gene_dim)
            ) for _ in range(num_experts)
        ])

    def forward(self, x, meta):
        temperature = 0.3
        img_emb = self.encoder(x)        # (B, 512)
        meta_emb = self.meta_encoder(meta)  # (B, 64)

        combined = torch.cat([img_emb, meta_emb], dim=1)

        gate_logits = self.gate(combined)
        gate_weights = torch.softmax(gate_logits/temperature, dim=1)
        expert_input = torch.cat([img_emb, meta_emb], dim=1)
        expert_outputs = torch.stack(
            [expert(expert_input) for expert in self.experts],
            dim=1
        )  # (B, E, G)

        output = (gate_weights.unsqueeze(-1) * expert_outputs).sum(dim=1)

        return output, gate_weights

In [20]:
class Baseline(nn.Module):
    def __init__(self, gene_dim=300):
        super().__init__()

        self.encoder = resnet18(pretrained=True)
        self.encoder.fc = nn.Identity()

        self.meta_encoder = MetadataEncoder()

        self.head = nn.Sequential(
            nn.Linear(512+64, 512),
            nn.ReLU(),
            nn.Linear(512, gene_dim)
        )

    def forward(self, x, meta):
        img = self.encoder(x)
        meta = self.meta_encoder(meta)
        x = torch.cat([img, meta], dim=1)
        return self.head(x)

In [9]:
def load_balancing_loss(gate_weights):
    # mean usage per expert
    usage = gate_weights.mean(dim=0)

    # encourage uniform usage
    target = torch.ones_like(usage) / len(usage)

    return ((usage - target) ** 2).sum()

In [10]:
import torch

def pearson_loss(pred, target):
    pred = pred - pred.mean(dim=1, keepdim=True)
    target = target - target.mean(dim=1, keepdim=True)

    pred_norm = torch.norm(pred, dim=1)
    target_norm = torch.norm(target, dim=1)

    # avoid zero division
    valid = (pred_norm > 1e-6) & (target_norm > 1e-6)

    corr = torch.zeros_like(pred_norm)

    corr[valid] = (
        (pred[valid] * target[valid]).sum(dim=1) /
        (pred_norm[valid] * target_norm[valid])
    )

    loss = 1 - corr.mean()
    return loss

def pearson_metric(pred, target):
    return 1 - pearson_loss(pred, target)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = MoE().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
mse_loss = nn.MSELoss()

c:\Users\dhanu\Documents\preliminary-gsoc\histMOE\venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\dhanu\Documents\preliminary-gsoc\histMOE\venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [ ]:
MODEL_DIR = os.path.join(BASE_DIR, "models")
os.makedirs(MODEL_DIR, exist_ok=True)
save_path = os.path.join(MODEL_DIR, "best_moe_model.pth")
best_val_loss = float("inf")

for epoch in range(10):

    #  TRAIN 
    model.train()
    train_loss = 0

    for images, genes, meta, labels in tqdm(train_loader):

        images = images.to(device)
        genes  = genes.to(device)
        meta   = meta.to(device)

        optimizer.zero_grad()

        pred, gate = model(images, meta)

        loss_main = mse_loss(pred, genes)
        loss_lb   = load_balancing_loss(gate)
        loss_corr = pearson_loss(pred, genes)

        loss = loss_main + 0.1 * loss_corr + 0.1 * loss_lb

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    #  VALIDATION 
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for images, genes, meta, labels in val_loader:

            images = images.to(device)
            genes  = genes.to(device)
            meta   = meta.to(device)

            pred, gate = model(images, meta)

            loss_main = mse_loss(pred, genes)
            loss_lb   = load_balancing_loss(gate)
            loss_corr = pearson_loss(pred, genes)
            loss = loss_main + 0.1 * loss_corr + 0.1 * loss_lb

            val_loss += loss.item()

    val_loss /= len(val_loader)

    print(f"Epoch {epoch} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    print("Val Pearson:", pearson_metric(pred, genes).item())

    #  SAVE BEST MODEL 
    if val_loss < best_val_loss:
        best_val_loss = val_loss

        torch.save({
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "epoch": epoch,
            "val_loss": val_loss
        }, save_path)

        print("✅ Best model saved!")

  0%|          | 0/360 [00:00<?, ?it/s]C:\Users\dhanu\AppData\Local\Temp\ipykernel_3120\467571399.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(image, dtype=torch.float32),
100%|██████████| 360/360 [00:56<00:00,  6.36it/s]


Epoch 0 | Train Loss: 0.1480 | Val Loss: 0.2019
Val Pearson: 0.2484338879585266
✅ Best model saved!


100%|██████████| 360/360 [00:53<00:00,  6.74it/s]


Epoch 1 | Train Loss: 0.1318 | Val Loss: 0.2042
Val Pearson: 0.25376570224761963


100%|██████████| 360/360 [00:57<00:00,  6.31it/s]


...

Epoch 7 | Train Loss: 0.1088 | Val Loss: 0.2089
Val Pearson: 0.24182873964309692


100%|██████████| 360/360 [00:56<00:00,  6.43it/s]


Epoch 8 | Train Loss: 0.1074 | Val Loss: 0.2077
Val Pearson: 0.2540147304534912


100%|██████████| 360/360 [00:53<00:00,  6.69it/s]


Epoch 9 | Train Loss: 0.1060 | Val Loss: 0.2074
Val Pearson: 0.246557354927063


now :

- Metadata-conditioned experts
- Temperature-controlled gating
- Entropy + 0.3*load balancing regularization
- hvg - 1000

In [12]:
def entropy_loss(gate_weights):
    return -(gate_weights * torch.log(gate_weights + 1e-8)).sum(dim=1).mean()

In [13]:
patience = 3
epochs_no_improve = 0
min_delta = 1e-4  # minimum improvement to count

In [16]:
MODEL_DIR = os.path.join(BASE_DIR, "models")
os.makedirs(MODEL_DIR, exist_ok=True)
save_path = os.path.join(MODEL_DIR, "best_moe_model.pth")
best_val_loss = float("inf")

for epoch in range(10):

    #  TRAIN 
    model.train()
    train_loss = 0

    for images, genes, meta, labels in tqdm(train_loader):

        images = images.to(device)
        genes  = genes.to(device)
        meta   = meta.to(device)
        labels = labels.to(device) 
        optimizer.zero_grad()

        pred, gate = model(images, meta)

        loss_main = mse_loss(pred, genes)
        loss_lb   = load_balancing_loss(gate)
        loss_corr = pearson_loss(pred, genes)

        import torch.nn.functional as F
        expert_targets = F.one_hot(labels, num_classes=2).float()
        loss_expert = F.mse_loss(gate, expert_targets)

        loss = (
            loss_main
            + 0.1 * loss_corr
            + 0.3 * loss_lb
            + 0.05 * entropy_loss(gate)
            + 0.2 * loss_expert   # NEW
        )

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)
    if epoch == 0:  # just first epoch to avoid spam
        for c in [0, 1]:
            mask = (labels == c)
            if mask.sum() > 0:
                print(f"Train Class {c} gate:", gate[mask].mean(dim=0))

    #  VALIDATION 
    model.eval()
    val_loss = 0

    all_gates = []
    all_labels = []

    with torch.no_grad():
        for images, genes, meta, labels in val_loader:

            images = images.to(device)
            genes  = genes.to(device)
            meta   = meta.to(device)
            labels = labels.to(device)
            pred, gate = model(images, meta)

            all_gates.append(gate.cpu())
            all_labels.append(labels.cpu())

            loss_main = mse_loss(pred, genes)
            loss_lb   = load_balancing_loss(gate)
            loss_corr = pearson_loss(pred, genes)

            import torch.nn.functional as F
            expert_targets = F.one_hot(labels, num_classes=2).float()
            loss_expert = F.mse_loss(gate, expert_targets)

            loss = (
                loss_main
                + 0.1 * loss_corr
                + 0.3 * loss_lb
                + 0.05 * entropy_loss(gate)
                + 0.2 * loss_expert   # NEW
            )

            val_loss += loss.item()

    val_loss /= len(val_loader)

    print(f"Epoch {epoch} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    print("Val Pearson:", pearson_metric(pred, genes).item())

    all_gates = torch.cat(all_gates, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    print("Overall gate:", all_gates.mean(dim=0))

    for c in [0, 1]:
        mask = (all_labels == c)
        print(f"Class {c} gate:", all_gates[mask].mean(dim=0))

    #  SAVE BEST MODEL 
    if val_loss < best_val_loss - min_delta:
        best_val_loss = val_loss
        epochs_no_improve = 0

        torch.save({
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "epoch": epoch,
            "val_loss": val_loss
        }, save_path)

        print(" Best model saved!")

    else:
        epochs_no_improve += 1
        print(f"No improvement for {epochs_no_improve} epoch(s)")
    
    if epochs_no_improve >= patience:
        print(f"\n Early stopping triggered at epoch {epoch}")
        break

100%|██████████| 360/360 [00:46<00:00,  7.68it/s]


Train Class 0 gate: tensor([0.9987, 0.0013], device='cuda:0', grad_fn=<MeanBackward1>)
Train Class 1 gate: tensor([0.0244, 0.9756], device='cuda:0', grad_fn=<MeanBackward1>)
Epoch 0 | Train Loss: 0.1752 | Val Loss: 0.3124
Val Pearson: 0.005989491939544678
Overall gate: tensor([0.6184, 0.3816])
Class 0 gate: tensor([0.9563, 0.0437])
Class 1 gate: tensor([0.5002, 0.4998])
 Best model saved!


100%|██████████| 360/360 [00:43<00:00,  8.37it/s]


Epoch 1 | Train Loss: 0.1491 | Val Loss: 0.3262
Val Pearson: 0.0035222768783569336
Overall gate: tensor([0.6861, 0.3139])
Class 0 gate: tensor([0.9934, 0.0066])
Class 1 gate: tensor([0.5786, 0.4214])
No improvement for 1 epoch(s)


100%|██████████| 360/360 [00:42<00:00,  8.55it/s]


Epoch 2 | Train Loss: 0.1430 | Val Loss: 0.3253
Val Pearson: 0.0023123621940612793
Overall gate: tensor([0.5135, 0.4865])
Class 0 gate: tensor([0.9975, 0.0025])
Class 1 gate: tensor([0.3443, 0.6557])
No improvement for 2 epoch(s)


100%|██████████| 360/360 [00:41<00:00,  8.57it/s]


Epoch 3 | Train Loss: 0.1371 | Val Loss: 0.3353
Val Pearson: 0.004582524299621582
Overall gate: tensor([0.7241, 0.2759])
Class 0 gate: tensor([9.9921e-01, 7.9416e-04])
Class 1 gate: tensor([0.6280, 0.3720])
No improvement for 3 epoch(s)

 Early stopping triggered at epoch 3


In [17]:
print(genes.std(axis=0).mean())

tensor(0.0068, device='cuda:0')


with hvg = 300

In [16]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = MoE(gene_dim=150).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
mse_loss = nn.MSELoss()

c:\Users\dhanu\Documents\preliminary-gsoc\histMOE\venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\dhanu\Documents\preliminary-gsoc\histMOE\venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [17]:
MODEL_DIR = os.path.join(BASE_DIR, "models")
os.makedirs(MODEL_DIR, exist_ok=True)
save_path = os.path.join(MODEL_DIR, "best_moe_model.pth")
best_val_loss = float("inf")

for epoch in range(10):

    #  TRAIN 
    model.train()
    train_loss = 0

    for images, genes, meta, labels in tqdm(train_loader):

        images = images.to(device)
        genes  = genes.to(device)
        meta   = meta.to(device)
        labels = labels.to(device) 
        optimizer.zero_grad()

        pred, gate = model(images, meta)

        loss_main = mse_loss(pred, genes)
        loss_lb   = load_balancing_loss(gate)
        loss_corr = pearson_loss(pred, genes)

        import torch.nn.functional as F
        expert_targets = F.one_hot(labels, num_classes=2).float()
        loss_expert = F.mse_loss(gate, expert_targets)

        loss = (
            loss_main
            + 0.1 * loss_corr
            + 0.3 * loss_lb
            + 0.05 * entropy_loss(gate)
            + 0.2 * loss_expert   # NEW
        )

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)
    if epoch == 0:  # just first epoch to avoid spam
        for c in [0, 1]:
            mask = (labels == c)
            if mask.sum() > 0:
                print(f"Train Class {c} gate:", gate[mask].mean(dim=0))

    #  VALIDATION 
    model.eval()
    val_loss = 0

    all_gates = []
    all_labels = []

    with torch.no_grad():
        for images, genes, meta, labels in val_loader:

            images = images.to(device)
            genes  = genes.to(device)
            meta   = meta.to(device)
            labels = labels.to(device)
            pred, gate = model(images, meta)

            all_gates.append(gate.cpu())
            all_labels.append(labels.cpu())

            loss_main = mse_loss(pred, genes)
            loss_lb   = load_balancing_loss(gate)
            loss_corr = pearson_loss(pred, genes)

            import torch.nn.functional as F
            expert_targets = F.one_hot(labels, num_classes=2).float()
            loss_expert = F.mse_loss(gate, expert_targets)

            loss = (
                loss_main
                + 0.1 * loss_corr
                + 0.3 * loss_lb
                + 0.05 * entropy_loss(gate)
                + 0.2 * loss_expert   # NEW
            )

            val_loss += loss.item()

    val_loss /= len(val_loader)

    print(f"Epoch {epoch} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    print("Val Pearson:", pearson_metric(pred, genes).item())

    all_gates = torch.cat(all_gates, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    print("Overall gate:", all_gates.mean(dim=0))

    for c in [0, 1]:
        mask = (all_labels == c)
        print(f"Class {c} gate:", all_gates[mask].mean(dim=0))

    #  SAVE BEST MODEL 
    if val_loss < best_val_loss - min_delta:
        best_val_loss = val_loss
        epochs_no_improve = 0

        torch.save({
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "epoch": epoch,
            "val_loss": val_loss
        }, save_path)

        print(" Best model saved!")

    else:
        epochs_no_improve += 1
        print(f"No improvement for {epochs_no_improve} epoch(s)")
    
    if epochs_no_improve >= patience:
        print(f"\n Early stopping triggered at epoch {epoch}")
        break

100%|██████████| 360/360 [00:40<00:00,  8.78it/s]


Train Class 0 gate: tensor([9.9996e-01, 3.6555e-05], device='cuda:0', grad_fn=<MeanBackward1>)
Train Class 1 gate: tensor([0.0052, 0.9948], device='cuda:0', grad_fn=<MeanBackward1>)
Epoch 0 | Train Loss: 0.2205 | Val Loss: 0.4027
Val Pearson: 0.0024780631065368652
Overall gate: tensor([0.5722, 0.4278])
Class 0 gate: tensor([0.9926, 0.0074])
Class 1 gate: tensor([0.4252, 0.5748])
 Best model saved!


100%|██████████| 360/360 [00:38<00:00,  9.43it/s]


Epoch 1 | Train Loss: 0.1925 | Val Loss: 0.4002
Val Pearson: 0.0008875727653503418
Overall gate: tensor([0.6409, 0.3591])
Class 0 gate: tensor([0.9963, 0.0037])
Class 1 gate: tensor([0.5167, 0.4833])
 Best model saved!


100%|██████████| 360/360 [00:39<00:00,  9.07it/s]


...

Epoch 3 | Train Loss: 0.1676 | Val Loss: 0.4367
Val Pearson: 0.000579833984375
Overall gate: tensor([0.4519, 0.5481])
Class 0 gate: tensor([0.9985, 0.0015])
Class 1 gate: tensor([0.2608, 0.7392])
No improvement for 1 epoch(s)


100%|██████████| 360/360 [00:36<00:00,  9.77it/s]


Epoch 4 | Train Loss: 0.1599 | Val Loss: 0.4028
Val Pearson: 0.0006150007247924805
Overall gate: tensor([0.5828, 0.4172])
Class 0 gate: tensor([9.9989e-01, 1.1010e-04])
Class 1 gate: tensor([0.4370, 0.5630])
No improvement for 2 epoch(s)


 34%|███▍      | 122/360 [00:12<00:25,  9.42it/s]


KeyboardInterrupt: 

In [18]:
print(pred.mean(), pred.std())
print(genes.mean(), genes.std())

tensor(0.1859, device='cuda:0', grad_fn=<MeanBackward0>) tensor(0.3853, device='cuda:0', grad_fn=<StdBackward0>)
tensor(0.1686, device='cuda:0') tensor(0.4726, device='cuda:0')


In [24]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = Baseline(gene_dim=150).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
mse_loss = nn.MSELoss()

In [26]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

def train_baseline(
    model,
    train_loader,
    val_loader,
    device,
    epochs=10,
    lr=1e-4,
    save_path="best_baseline.pth",
    patience=3
):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    mse_loss = nn.MSELoss()

    best_val_loss = float("inf")
    epochs_no_improve = 0
    min_delta = 1e-4

    model.to(device)

    for epoch in range(epochs):

        # ================= TRAIN =================
        model.train()
        train_loss = 0

        for images, genes, meta, labels in tqdm(train_loader):

            images = images.to(device)
            genes  = genes.to(device)
            meta   = meta.to(device)

            optimizer.zero_grad()

            pred = model(images, meta)

            loss_main = mse_loss(pred, genes)
            loss_corr = pearson_loss(pred, genes)

            loss = loss_main + 0.1 * loss_corr

            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)

        # ================= VALIDATION =================
        model.eval()
        val_loss = 0

        all_preds = []
        all_genes = []

        with torch.no_grad():
            for images, genes, meta, labels in val_loader:

                images = images.to(device)
                genes  = genes.to(device)
                meta   = meta.to(device)

                pred = model(images, meta)

                loss_main = mse_loss(pred, genes)
                loss_corr = pearson_loss(pred, genes)

                loss = loss_main + 0.1 * loss_corr

                val_loss += loss.item()

                all_preds.append(pred.cpu())
                all_genes.append(genes.cpu())

        val_loss /= len(val_loader)

        all_preds = torch.cat(all_preds, dim=0)
        all_genes = torch.cat(all_genes, dim=0)

        val_pearson = pearson_metric(all_preds, all_genes).item()

        print(f"\nEpoch {epoch}")
        print(f"Train Loss: {train_loss:.4f}")
        print(f"Val Loss: {val_loss:.4f}")
        print(f"Val Pearson: {val_pearson:.4f}")

        # ================= SAVE + EARLY STOP =================
        if val_loss < best_val_loss - min_delta:
            best_val_loss = val_loss
            epochs_no_improve = 0

            torch.save({
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "epoch": epoch,
                "val_loss": val_loss
            }, save_path)

            print("✅ Best model saved!")

        else:
            epochs_no_improve += 1
            print(f"No improvement for {epochs_no_improve} epoch(s)")

        if epochs_no_improve >= patience:
            print(f"\n🛑 Early stopping at epoch {epoch}")
            break

train_baseline(model,
    train_loader,
    val_loader,
    device,
    epochs=10,
    lr=1e-4,
    save_path="baseline_best_baseline.pth",
    patience=3)

100%|██████████| 360/360 [00:48<00:00,  7.50it/s]



Epoch 0
Train Loss: 0.1899
Val Loss: 0.2497
Val Pearson: 0.0987
✅ Best model saved!


100%|██████████| 360/360 [00:47<00:00,  7.61it/s]



Epoch 1
Train Loss: 0.1719
Val Loss: 0.2390
Val Pearson: 0.0966
✅ Best model saved!


100%|██████████| 360/360 [00:47<00:00,  7.62it/s]



Epoch 2
Train Loss: 0.1635
Val Loss: 0.2524
Val Pearson: 0.0881
No improvement for 1 epoch(s)


100%|██████████| 360/360 [00:47<00:00,  7.59it/s]



Epoch 3
Train Loss: 0.1501
Val Loss: 0.2746
Val Pearson: 0.0793
No improvement for 2 epoch(s)


100%|██████████| 360/360 [00:47<00:00,  7.55it/s]



Epoch 4
Train Loss: 0.1409
Val Loss: 0.2759
Val Pearson: 0.0779
No improvement for 3 epoch(s)

🛑 Early stopping at epoch 4


In [27]:
print(pred.mean(), pred.std())
print(genes.mean(), genes.std())

tensor(0.1859, device='cuda:0', grad_fn=<MeanBackward0>) tensor(0.3853, device='cuda:0', grad_fn=<StdBackward0>)
tensor(0.1857, device='cuda:0') tensor(0.4673, device='cuda:0')


In [24]:
import matplotlib.pyplot as plt

def plot_gene_prediction(gt, pred):
    plt.figure(figsize=(10,4))
    
    plt.plot(gt[:100], label="Ground Truth")
    plt.plot(pred[:100], label="Prediction")
    
    plt.legend()
    plt.title("Gene Prediction (first 100 genes)")
    plt.show()

In [25]:
def scatter_plot(gt, pred):
    plt.figure(figsize=(5,5))
    plt.scatter(gt, pred, alpha=0.3)
    plt.xlabel("Ground Truth")
    plt.ylabel("Prediction")
    plt.title("Gene Prediction Scatter")
    plt.show()

In [28]:
import numpy as np

def pearson(gt, pred):
    return np.corrcoef(gt, pred)[0,1]

In [29]:
import random

def test_random_samples(model, dataset, device, num_samples=5):
    model.eval()

    indices = random.sample(range(len(dataset)), num_samples)

    for idx in indices:
        image, gene, meta, label = dataset[idx]

        image = image.unsqueeze(0).to(device)
        meta  = meta.unsqueeze(0).to(device)

        with torch.no_grad():
            pred, gate = model(image, meta)

        pred = pred.cpu().numpy()[0]
        gt   = gene.numpy()

        print(f"\nSample {idx}")
        print("Label:", label.item())
        print("Gate:", gate.cpu().numpy())

        print("Pearson:", pearson(gt, pred))

        plot_gene_prediction(gt, pred)
        scatter_plot(gt, pred)

In [30]:
def show_patch(image_tensor):
    img = image_tensor.permute(1,2,0).numpy()
    plt.imshow(img)
    plt.axis("off")
    plt.show()

In [31]:
import numpy as np

def pearson(gt, pred):
    return np.corrcoef(gt, pred)[0,1]

In [32]:
def inspect_sample(model, dataset, idx, device):
    model.eval()

    image, gene, meta, label = dataset[idx]

    show_patch(image)

    image = image.unsqueeze(0).to(device)
    meta  = meta.unsqueeze(0).to(device)

    with torch.no_grad():
        pred, gate = model(image, meta)

    pred = pred.cpu().numpy()[0]
    gt   = gene.numpy()

    print("Label:", label.item())
    print("Gate:", gate.cpu().numpy())
    print("Pearson:", pearson(gt, pred))

    plot_gene_prediction(gt, pred)

In [33]:
def plot_gate_distribution(all_gates, all_labels):
    import matplotlib.pyplot as plt

    for c in [0,1]:
        mask = (all_labels == c)
        plt.hist(all_gates[mask, 0], alpha=0.5, label=f"Class {c}")

    plt.legend()
    plt.title("Expert 0 usage distribution")
    plt.show()